### 와인 데이터 파이토치로 분류

In [28]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset

from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay


In [18]:
df = pd.read_csv(r"D:\hanyeowon\Coding\pythonworks\deep\data\wine.csv")
df

,Wine,Alcohol,Malic.acid,Ash,Acl,Mg,Phenols,Flavanoids,Nonflavanoid.phenols,Proanth,Color.int,Hue,OD,Proline
0,1,14.23,1.71,2.43,15.6,127,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065
1,1,13.20,1.78,2.14,11.2,100,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050
2,1,13.16,2.36,2.67,18.6,101,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185
3,1,14.37,1.95,2.50,16.8,113,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480
4,1,13.24,2.59,2.87,21.0,118,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
173,3,13.71,5.65,2.45,20.5,95,1.68,0.61,0.52,1.06,7.70,0.64,1.74,740
174,3,13.40,3.91,2.48,23.0,102,1.80,0.75,0.43,1.41,7.30,0.70,1.56,750
175,3,13.27,4.28,2.26,20.0,120,1.59,0.69,0.43,1.35,10.20,0.59,1.56,835
176,3,13.17,2.59,2.37,20.0,120,1.65,0.68,0.53,1.46,9.30,0.60,1.62,840


In [19]:
y = df['Wine'].values-1
X = df.drop('Wine', axis=1).values

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X = scaler.fit_transform(X)

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0)
X_train, X_test, y_train, y_test

(array([[ 0.78980621,  0.68550197,  0.70724686, ...,  0.01119018,
          1.05695159,  0.3124203 ],
        [-0.49486935,  0.11099756, -0.60867587, ..., -0.99789773,
         -1.45719662, -0.16525376],
        [-1.28543893, -1.11880095, -0.24314178, ...,  0.14281034,
          0.73208974,  0.44298455],
        ...,
        [-0.71721705, -0.65201611, -0.64522928, ...,  0.44992405,
          0.49197446, -1.27982657],
        [ 1.1109751 , -0.58917969, -0.90110314, ..., -0.20817676,
          1.01457831,  0.75824943],
        [ 1.43214399,  0.15588072,  0.41481959, ..., -1.48050498,
         -1.27357906, -0.27671104]], shape=(133, 13)),
 array([[ 9.13332708e-01, -5.98156324e-01, -4.25908823e-01,
         -9.29365181e-01,  1.28198515e+00,  4.88531085e-01,
          8.74184283e-01, -1.22360954e+00,  5.09876168e-02,
          3.42556546e-01, -1.64303370e-01,  8.30960739e-01,
          9.97086458e-01],
        [-2.60169009e-01,  2.99506821e-01,  4.14819587e-01,
          7.52230776e-01,  8.

In [20]:
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.int64)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.int64)

In [21]:
train_dataset = TensorDataset(X_train, y_train)
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True)

test_dataset = TensorDataset(X_test, y_test)
test_dataloader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [22]:
# X의 열과 y의 개수 확인
print(X.shape)
print(pd.unique(y))

(178, 13)
[0 1 2]


In [23]:
class CarEvaluationDense(nn.Module):
    def __init__(self):
        super(CarEvaluationDense, self).__init__()
        self.fc1 = nn.Linear(13, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 3) 
    
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x)
        return x

model = CarEvaluationDense()

In [24]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [26]:
train_losses = []
test_accuracies = []

# Training loop
num_epochs = 20
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_dataloader:
        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # Backward pass and optimize
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    # Calculate average loss over an epoch
    train_losses.append(running_loss / len(train_dataloader))
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in test_dataloader:
            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    test_accuracies.append(accuracy)

    print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {train_losses[-1]:.4f}, Accuracy: {accuracy:.2f}%")

print("Training complete.")

Epoch 1/20, Loss: 0.0778, Accuracy: 100.00%
Epoch 2/20, Loss: 0.0684, Accuracy: 100.00%
Epoch 3/20, Loss: 0.0752, Accuracy: 100.00%
Epoch 4/20, Loss: 0.0647, Accuracy: 100.00%
Epoch 5/20, Loss: 0.0565, Accuracy: 100.00%
Epoch 6/20, Loss: 0.0511, Accuracy: 100.00%
Epoch 7/20, Loss: 0.0469, Accuracy: 100.00%
Epoch 8/20, Loss: 0.0525, Accuracy: 100.00%
Epoch 9/20, Loss: 0.0454, Accuracy: 100.00%
Epoch 10/20, Loss: 0.0375, Accuracy: 100.00%
Epoch 11/20, Loss: 0.0691, Accuracy: 100.00%
Epoch 12/20, Loss: 0.0372, Accuracy: 100.00%
Epoch 13/20, Loss: 0.0348, Accuracy: 100.00%
Epoch 14/20, Loss: 0.0321, Accuracy: 100.00%
Epoch 15/20, Loss: 0.0298, Accuracy: 100.00%
Epoch 16/20, Loss: 0.0323, Accuracy: 100.00%
Epoch 17/20, Loss: 0.0273, Accuracy: 100.00%
Epoch 18/20, Loss: 0.0239, Accuracy: 100.00%
Epoch 19/20, Loss: 0.0224, Accuracy: 100.00%
Epoch 20/20, Loss: 0.0228, Accuracy: 100.00%
Training complete.


In [29]:
# Evaluation
model.eval()
all_labels = []
all_predictions = []
with torch.no_grad():
    for inputs, labels in test_dataloader:
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        all_labels.extend(labels.cpu().numpy())
        all_predictions.extend(predicted.cpu().numpy())

# Convert to numpy arrays
all_labels = np.array(all_labels)
all_predictions = np.array(all_predictions)

# Calculate metrics
conf_matrix = confusion_matrix(all_labels, all_predictions)
f1 = f1_score(all_labels, all_predictions, average='weighted')
precision = precision_score(all_labels, all_predictions, average='weighted')
recall = recall_score(all_labels, all_predictions, average='weighted')

# Calculate specificity for each class
specificity = []
for i in range(conf_matrix.shape[0]):
    tn = conf_matrix.sum() - (conf_matrix[i, :].sum() + conf_matrix[:, i].sum() - conf_matrix[i, i])
    fp = conf_matrix[:, i].sum() - conf_matrix[i, i]
    specificity.append(tn / (tn + fp))

print(f'Confusion Matrix:\n{conf_matrix}')
print(f'F1 Score: {f1:.2f}')
print(f'Precision: {precision:.2f}')
print(f'Recall: {recall:.2f}')
print(f'Specificity: {np.mean(specificity):.2f}')

Confusion Matrix:
[[16  0  0]
 [ 0 21  0]
 [ 0  0  8]]
F1 Score: 1.00
Precision: 1.00
Recall: 1.00
Specificity: 1.00
